# Week 1-3

In the first 6 Chapters I learned:

- Variables and data types
- Basic operations
- Input and output
- Type conversion
- Lists and tuples
- Dictionaries and sets
- Functions
- Control flow (if statements, loops)
- Basic File Operations

Using this knowledge, I want to build the first part of my Python Portfolio.<br>
The Portfolio itself should eventually become a flexible tool to automate tasks within a machine learning workflow. <br>The first part consists of:

- Sourcing:  <br>Tha application should read a text file containing a list of urls, the application should iterate through the list and download the files from the internet or a given path, into the sourcing folder.  <br> At the end of the preocess, the application should print out a summary of every step. (url / path, timestamp,  filename, size, time taken to download).  <br>For the failed downloads, the user can choose wehther to create a separate list of the config-file for another attempt. It should ask whether the user wants to retry the failed downloads before next initiation with the original list starts, should the user chooses no.

- Data cleansing: <br> The application should read a text file containing lists of file selection (file name filter), the cleansing function(s), it should apply to the files. The cleaned version should be saved in a separate directory. A summary should be printed out for every operation done for each file.

- Analyze:  <br>The user can choose from a list of files (clean data) to display different basic statistics of the data. And the end of the operation, the user can choose whether to choose another file to look at or quit.
 <br>
From ChatGPT:

🔹 1. Basic Structure and Metadata

- Shape: Number of rows and columns (df.shape)
- Column names and types (df.dtypes)
- Index: Is it meaningful or just default?
- Sample data: View the first few rows (df.head())

🔹 2. Variable (Feature) Information

For each column:
- Name
- Type (Numerical, Categorical, Text, Date/Time, Boolean)
- Unique values (df['col'].unique())
- Missing values (df.isnull().sum())

🔹 3. Descriptive Statistics

Numerical features:
- Mean, median, min, max, std dev (df.describe())
- Distributions (histograms, box plots)
- Categorical features:
- Frequency counts (df['col'].value_counts())
- Mode, cardinality (number of unique values)

🔹 4. Data Quality Checks

- Missing data: Amount, location, and patterns
- Duplicates: Rows or entries (df.duplicated())
- Outliers: Use IQR, Z-score, or visual tools (box plots)
- Inconsistent data: Typos, mixed types in one column

🔹 5. Relationships and Correlations

- Correlation matrix for numerical variables
- Cross-tabulations or groupings for categorical vs numerical
- Pairplots or scatter matrix for visual inspection

🔹 6. Target Variable Understanding (if doing supervised learning)

- Type (Regression or Classification)
- Distribution and balance of classes or range of values
- Any leakage or target-related data issues

🔹 7. Contextual Understanding

- Source of the data
- What each column means (check data dictionary or metadata)
- Domain relevance (is something unusual, or expected?)
- Timeframes involved (especially for time series data)

Optional but Helpful

- Units: Are amounts in USD? Lengths in meters?
- Encoding: Are categories encoded numerically?
- Timezones: For datetime fields


## Initiation

Initiate global variables + Imports

In [5]:
import urllib.request
import shutil
import os
import pandas as pd
import numpy as np

#Constants
# Get the current working directory

global WORKING_DIRECTORY, SOURCE_DIRECTORY, PREPROCESS_DIRECTORY, TESTS_DIRECTORY,OUTPUT_DIRECTORY, RAW_DATAFILES, SUPPORTED_FILEFORMATS

WORKING_DIRECTORY = os.getcwd()
SOURCE_DIRECTORY = WORKING_DIRECTORY+ "\\2_Sourcing\\"
PREPROCESS_DIRECTORY = WORKING_DIRECTORY+ "\\3_Data_Preprocessing\\"
TESTS_DIRECTORY = WORKING_DIRECTORY+ "\\4_Tests\\"
OUTPUT_DIRECTORY = WORKING_DIRECTORY+ "\\8_Output\\"

SUPPORTED_FILEFORMATS={"CSV":pd.read_csv,"TXT":pd.read_fwf,"JSON":pd.read_json,"XML":pd.read_xml,"XLSX":pd.read_excel,"XLS":pd.read_excel}

print("The current working directory is:", WORKING_DIRECTORY)

#Raw Data Files
RAW_DATAFILES={}

The current working directory is: h:\Documents\My Training\CAS\CAS_DLS


### 2. Read Config Function

Should read each line from a cfg file (text) and store it in a list.

In [6]:
def read_config(file_path,targetlist):
###
# Filepath: Location of the editable source.cfg file, 1 line per file in the following format [url / folder path],[filename],[target-folder]
# Targetlist: can be sourceCFGList or missCFGList
###
    try:
        with open(file_path, 'r') as file:
            config = file.read()
            for line in config.splitlines():
                #print(line.strip())
                x,y,z=line.strip().split(',')
                targetlist.append([x,y,z])
            print("Configuration loaded:")
            print(config)
    except FileNotFoundError:
        print(f"Configuration file '{file_path}' not found.")
        return None

### 4. Write Config Function

Write back the missing files into the miss.cfg file for the next run.

- Parameters: config file path, ArrayList (url, filename, destination)

In [7]:
def write_config(file_path, data):
###
# File_path: Location of the editable source.cfg file, 1 line per file in the following format [url / folder path],[filename],[target-folder]
# data: list of sourcable items to store
###
    try:
        with open(file_path, 'w') as file:
            for line in data:
                file.write(','.join(line) + '\n')
        print(f"Configuration saved to '{file_path}'")
    except Exception as e:
        print(f"Error writing configuration file: {e}")
        return None

### Download File
Retrieve a file from www and save it to the destination path

In [8]:
def download_file(url, filename, destination):
    ## exception handling moved to upper layer
    #try:
        urllib.request.urlretrieve(url, WORKING_DIRECTORY + destination + filename)
        print(f"File downloaded successfully: {destination}")
    #except Exception as e:
    #    print(f"Error downloading file: {e}")
    #    return None


### Copy File Function

Copy file from UNC-Path and save it to destination.

In [9]:
def copy_file(source, destination):
    ## exception handling moved to upper layer
    #try:
        shutil.copy(source, WORKING_DIRECTORY+ destination)
        print(f"File copied successfully from {source} to {destination}")
    #except Exception as e:
    #   print(f"Error copying file: {e}")

## Sourcing Function
Import files from different sources (web or network share) using a predefined config-file.<br>
If missing files from last iteration exists, automatically ask the user whether to continue with the import or retry missing files first.

In [10]:
def sourcing_process():
    #Source Items
    sourceCFG = SOURCE_DIRECTORY + "source.cfg"
    sourceCFGList = []

    #Missing items
    missCFG = SOURCE_DIRECTORY +"miss.cfg"
    missCFGList = []

    print("You chose Sourcing")

    def source_list(CFGList):
    ###
    # Iterate through a list and call download or copy file function according to the given sourcepath and call the write cfg for failed attempts.
    # Parameter list of Array ( URL, Filename, Destination)
    ###
        NewmissCFGList = []
        for sourceCFG in CFGList:
            url = str(sourceCFG[0])
            filename = sourceCFG[1]
            destination = sourceCFG[2]
            try:
                if url.lower().startswith("http"):
                    download_file(url, filename, destination)
                    #missCFGList.remove[sourceCFG] does not work while in the for loop
                else:
                    copy_file(url, destination)
                    #missCFGList.remove[sourceCFG] dito
            except Exception as e:
                print(f"Error sourcing file: {e}")
                NewmissCFGList.append([url,filename,destination])
        CFGList=[]
        if NewmissCFGList:
            write_config(missCFG, NewmissCFGList)
            raise RuntimeError("Not all items were sourced." + str(NewmissCFGList).join("\n"))

    #precheck whether missing items / file (bkp.cfg) exists
    if os.path.exists(missCFG):
        read_config(missCFG,missCFGList)
        #check whether list exists / not empty
        if missCFGList:
            response = input("There are open Items for download, do you whish to retry (y/n)?\n" + str(missCFGList).join("\n")).strip().lower()
            if response == 'y':
                try:
                    #Source all files listed in config
                    source_list(missCFGList)

                    #The following will be skipped if an error occures in the above function:
                    os.remove(missCFG)
                except Exception as e:
                    print("Retry unsuccessful.")
            else:
                response =input("Do you want to delete the open itmes (y/n)?")
                if response =='y':
                    os.remove(missCFG)
    
    #main sourcing process
    if os.path.exists(sourceCFG):
        response = input("Would you like to start sourcing the data? (y/n): ").strip().lower()
        if response == 'y':
            print("Starting data sourcing...")
            read_config(sourceCFG,sourceCFGList)
            if sourceCFGList:
                #print("example destination:"+sourceCFGList[1][2])
                source_list(sourceCFGList)
    else:
        print(f"Source configuration file '{sourceCFG}' not found.")

### Load File function
Load File into memory for processing.

In [ ]:
def loadfile(filename):
    #experiments:
    df=SUPPORTED_FILEFORMATS[filename.upper().split(".")[-1]](filename)

    #metadata
    print(df)
    print("head:")
    print(df.head())
    print("tail:")
    print(df.tail())
    print("info:")
    df.info()
    print("describe:")
    print(df.describe())
    print("columns:")
    print(df.columns)
    print("index:")
    print(df.index)

    #outliers
    for column in df.columns:
        try:
            print(df.describe()[column])
        except:
            print("No statistics for(str / object)  column: "+column)

    

## Data Load Function
Show list of available files and load it upon user's choice.

In [79]:
def dataload_process():
    print("You chose Data Load")
    rawlist = os.listdir(SOURCE_DIRECTORY)
    filteredlist = [x for x in rawlist if x.upper().split(".")[-1] in list(SUPPORTED_FILEFORMATS.keys())]
    print(filteredlist)
    possibleChoice =[]

    print("Please choose one of the following file to be loaded: ")
    for idx, file in enumerate(filteredlist):
        print(f"{idx}: {file}")
        possibleChoice.append(str(idx))

    possibleChoice.append("x")
    print("x: (exit)")

    inputMissing = True
    while inputMissing:
        response = input("")
        if response in possibleChoice:
            inputMissing=False
    
    if not response =="x":
        loadfile(SOURCE_DIRECTORY + filteredlist[int(response)])

# Main UX Loop

Iterate through the main processes with user prompts

In [80]:
if __name__ == "__main__":
    exit = False
    while exit != True:
        response = input("Please choose processing mode: \n 1. Sourcing \n 2. Load Data \n 3. Analyze\n 4. Results\n 5. Walkthrough\n 6. Exit")
        if response == '1':
           sourcing_process()
        elif response == '2':
            dataload_process()
        elif response == '6':
            exit = True
            print("Exiting the program.")

You chose Data Load
['su-q-01.04.00.12.xlsx', 'ts-q-01.04.02.01.30-P.csv', 'ts-x-16.02.01-01.csv']
Please choose one of the following file to be loaded: 
0: su-q-01.04.00.12.xlsx
1: ts-q-01.04.02.01.30-P.csv
2: ts-x-16.02.01-01.csv
x: (exit)
       year  week        date unit recent origin   value
0      2019     1  2019-01-02  adm   rall   oall  307923
1      2019     2  2019-01-09  adm   rall   oall  249573
2      2019     3  2019-01-16  adm   rall   oall  240276
3      2019     4  2019-01-23  adm   rall   oall  303711
4      2019     5  2019-01-30  adm   rall   oall  333087
...     ...   ...         ...  ...    ...    ...     ...
17325  2025    30  2025-07-23  scr   rnew    ous     453
17326  2025    31  2025-07-30  scr   rnew    ous     438
17327  2025    32  2025-08-06  scr   rnew    ous     449
17328  2025    33  2025-08-13  scr   rnew    ous     436
17329  2025    34  2025-08-20  scr   rnew    ous     456

[17330 rows x 7 columns]
head:
   year  week        date unit recent orig